# 08.8 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/08-deep-learning/08-cnns-fashion-mnist/solutions.ipynb)

In [1]:
import os, warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"        # hide TensorFlow's C++ start-up messages
warnings.filterwarnings("ignore")                # hide library deprecation notices
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers
keras.utils.set_random_seed(42)                  # same start every run (results still vary slightly by machine)
print("Keras", keras.__version__, "· backend", keras.backend.backend())

(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()
CLASSES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
           "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
x_train, x_test = x_train / 255.0, x_test / 255.0      # pixels 0-255  ->  0-1
print("train", x_train.shape, " test", x_test.shape)

import time

def dense_model():
    return keras.Sequential([
        keras.Input((28, 28, 1)),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ], name="dense")

def cnn_model():
    return keras.Sequential([
        keras.Input((28, 28, 1)),
        layers.Conv2D(16, 3, activation="relu"),
        layers.MaxPooling2D(2),
        layers.Conv2D(32, 3, activation="relu"),
        layers.MaxPooling2D(2),
        layers.Flatten(),
        layers.Dense(64, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ], name="cnn")

def train(model, epochs=8):
    """Compile, fit on the training set (10% held out for validation), return history and seconds."""
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    start = time.time()
    history = model.fit(x_train[..., None], y_train, epochs=epochs, batch_size=128,
                        validation_split=0.1, verbose=0)
    return history, time.time() - start

Keras 3.15.1 · backend tensorflow
train (60000, 28, 28)  test (10000, 28, 28)


## Exercise 1

| layer | output shape | parameters |
|---|---|---|
| Conv2D(32, 3) | 26 × 26 × 32 | 3·3·1·32 + 32 = 320 |
| MaxPooling2D | 13 × 13 × 32 | 0 |
| Conv2D(64, 3) | 11 × 11 × 64 | 3·3·32·64 + 64 = 18,496 |
| MaxPooling2D | 5 × 5 × 64 | 0 |
| Flatten | 1,600 | 0 |
| Dense(128) | 128 | 1,600·128 + 128 = 204,928 |
| Dense(10) | 10 | 128·10 + 10 = 1,290 |
| **total** | | **225,034** |

Most of the weights sit in the first dense layer, not in the convolutions.

In [2]:
big = keras.Sequential([keras.Input((28, 28, 1)),
    layers.Conv2D(32, 3, activation="relu"), layers.MaxPooling2D(2),
    layers.Conv2D(64, 3, activation="relu"), layers.MaxPooling2D(2),
    layers.Flatten(), layers.Dense(128, activation="relu"), layers.Dense(10, activation="softmax")])
print(f"{big.count_params():,}")

225,034


## Exercise 2

In [3]:
def cnn_dropout():
    return keras.Sequential([keras.Input((28, 28, 1)),
        layers.Conv2D(16, 3, activation="relu"), layers.MaxPooling2D(2),
        layers.Conv2D(32, 3, activation="relu"), layers.MaxPooling2D(2),
        layers.Flatten(), layers.Dropout(0.3),
        layers.Dense(64, activation="relu"), layers.Dense(10, activation="softmax")])

for name, build in [("no dropout", cnn_model), ("dropout 0.3", cnn_dropout)]:
    keras.utils.set_random_seed(42)
    h, secs = train(build(), epochs=15)
    tr, va = h.history["accuracy"][-1], h.history["val_accuracy"][-1]
    print(f"{name:<12} train {tr:.3f}  val {va:.3f}  gap {tr - va:+.3f}  ({secs:.0f} s)")

no dropout   train 0.930  val 0.910  gap +0.020  (47 s)


dropout 0.3  train 0.912  val 0.911  gap +0.001  (60 s)


Dropout narrows the train–validation gap: the network cannot lean on any single neuron, so it memorises less. Whether validation accuracy also rises after 15 epochs depends on the run; the smaller gap is the reliable effect.

## Exercise 3

In [4]:
def cnn_gap():
    return keras.Sequential([keras.Input((28, 28, 1)),
        layers.Conv2D(16, 3, activation="relu"), layers.MaxPooling2D(2),
        layers.Conv2D(32, 3, activation="relu"), layers.MaxPooling2D(2),
        layers.GlobalAveragePooling2D(),
        layers.Dense(64, activation="relu"), layers.Dense(10, activation="softmax")])

for name, build in [("Flatten", cnn_model), ("GlobalAveragePooling", cnn_gap)]:
    keras.utils.set_random_seed(42)
    m = build(); h, secs = train(m)
    print(f"{name:<21} params {m.count_params():>6,}  test {m.evaluate(x_test[..., None], y_test, verbose=0)[1]:.3f}")

Flatten               params 56,714  test 0.894


GlobalAveragePooling  params  7,562  test 0.786


Global average pooling turns each 5×5×32 block into just 32 numbers, so the first dense layer shrinks from 800·64 + 64 = 51,264 weights to 32·64 + 64 = 2,112. On these tiny 28×28 photos that throws away *where* in the image a pattern was, and accuracy drops. On large photos (08.9) global pooling is the standard choice.

## Exercise 4

In [5]:
keras.utils.set_random_seed(42)
cnn = cnn_model(); train(cnn)
probs = cnn.predict(x_test[..., None], verbose=0)
pred, conf = probs.argmax(1), probs.max(1)
right = pred == y_test
top2 = (np.argsort(probs, axis=1)[:, -2:] == y_test[:, None]).any(axis=1)
print(f"accuracy {right.mean():.3f}   top-2 accuracy {top2.mean():.3f}")
print(f"average confidence when right {conf[right].mean():.3f}, when wrong {conf[~right].mean():.3f}")
print(f"wrong predictions made with over 90% confidence: {(conf[~right] > 0.9).sum()} of {(~right).sum()}")

accuracy 0.894   top-2 accuracy 0.973
average confidence when right 0.928, when wrong 0.670
wrong predictions made with over 90% confidence: 141 of 1059


The model is less sure when it is wrong, which is useful: ShopWave can send low-confidence photos to a person and auto-tag the rest. But some mistakes are made with high confidence, so a confidence threshold reduces errors without removing them. Top-2 accuracy is much higher than top-1: offering "Shirt or T-shirt?" as two buttons would fix most of the remaining mix-ups.